# Flight Pricing Dataset — Data Analysis

## Objective

The goal of this analysis is to understand the flight pricing dataset,
identify data quality problems, understand the relationships between
features and flight price, and determine which preprocessing steps are
required before training a machine learning model.

## Dataset

The dataset contains information about flights including airline,
source, destination, travel class, flight duration, number of stops,
distance, passenger count, booking information, and price.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/flight_pricing_dataset.csv")

df.head()
df.shape
df.info()
df.describe(include="all)


missing = df.isnull().sum().sort_values(ascending=False)

missing

missing_percentage = (
    df.isnull().mean() * 100
).sort_values(ascending=False)

missing_percentage

plt.figure(figsize=(10, 6))

missing_percentage.plot(kind="bar")

plt.ylabel("Missing values (%)")
plt.xlabel("Feature")
plt.title("Missing Values by Feature")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Observation

Several features contain missing values. Therefore, missing-value
handling will be required before model training.

Numeric features can be filled using statistics calculated from the
training data, while categorical missing values can be represented
using an "Unknown" category.


### Price

The price column contains currency formatting and commas, so it cannot
be directly used as a numerical feature.

Example:

`Rs. 1,25,000`

needs to become:

`125000`

The cleaning pipeline therefore removes the currency prefix and commas
and converts the result to a numeric type.

df["Duration"].head(30)

df["Duration"].dropna().astype(str).head(50)

df["Duration"].dropna().astype(str).str.contains("min").sum()

df["Duration"].dropna().astype(str).str.contains("h").sum()


### Observation

Duration is represented using multiple formats.

For machine learning, these representations should be converted into
one consistent numerical unit.

I chose minutes as the standard unit because it preserves the duration
information while producing a numerical feature.



df["Total_Stops"].value_counts(dropna=False)

### Observation

The `Total_Stops` feature contains different textual representations
such as non-stop, `1 stop`, and `2 stops`.

These values represent the same underlying numerical concept, so they
are converted into the number of stops:

non-stop → 0  
1 stop → 1  
2 stops → 2  
...


# after applying the cleaning logic
df["Total_Stops"].value_counts(dropna=False)


df["Passenger_Count"].value_counts(dropna=False).head(20)

### Observation

Passenger count contains both numeric strings and number words.

For example:

`"5"` and `"five"` represent the same numerical value.

These representations are converted into integers so that the feature
can be used numerically.


Indigo
INDIGO
indigo

Air India
AIR INDIA
air india

### Observation

The airline feature contains inconsistent capitalization. These values
represent the same airline, so capitalization should not create
separate categories.

The values are normalized to lowercase.


print(sorted(df["Source"].dropna().unique()))
print(sorted(df["Destination"].dropna().unique()))

### Observation

Source and Destination contain multiple representations of the same
location.

For example:

Ahmedabad
Ahmedabad Airport
AMD

represent the same location.

Therefore, these representations are mapped to a canonical location
name.

The same mapping is applied to both Source and Destination.



df[["Departure_Time", "Arrival_Time"]].head(20)

### Observation

Departure and arrival times are represented using both 12-hour and
24-hour formats.

The values are converted into minutes after midnight.

For example:

8:10 PM → 1210 minutes
00:05 → 5 minutes


### Cyclical time representation

Time is cyclical. 23:59 and 00:01 are only two minutes apart, but
treating them as ordinary numbers would make them appear very far apart.

Therefore, departure and arrival times are represented using sine and
cosine transformations.

minutes = np.arange(0, 1440)

sin_values = np.sin(2 * np.pi * minutes / 1440)
cos_values = np.cos(2 * np.pi * minutes / 1440)

plt.figure(figsize=(10, 5))
plt.plot(minutes, sin_values, label="sin")
plt.plot(minutes, cos_values, label="cos")

plt.xlabel("Minutes after midnight")
plt.ylabel("Value")
plt.title("Cyclical Representation of Time")
plt.legend()
plt.grid()
plt.show()


## Departure Date

The raw departure date is converted into a datetime object.

Instead of passing the raw date string directly to the model, useful
components are extracted:

- Departure Year
- Departure Month
- Departure Day
- Departure Day of Year


df["Departure_Date"].dt.month.value_counts().sort_index().plot(kind="bar")

plt.xlabel("Month")
plt.ylabel("Number of flights")
plt.title("Flights by Departure Month")
plt.show()


